# Normalization and pre-tokenization

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
# !pip install datasets evaluate transformers[sentencepiece]

## Normalization

The normalization step involves some general cleanup, such as removing needless whitespace, lowercasing, and/or removing accents. If you're familiar with [Unicode normalization](http://www.unicode.org/reports/tr15/) (such as NFC or NFKC), this is also something the tokenizer may apply.


The 🤗 Transformers `tokenizer` has an attribute called `backend_tokenizer` that provides access to the underlying tokenizer from the 🤗 Tokenizers library:

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(type(tokenizer.backend_tokenizer))

<class 'tokenizers.Tokenizer'>


The `normalizer` attribute of the `tokenizer` object has a `normalize_str()` method that we can use to see how the normalization is performed:

In [2]:
print(tokenizer.backend_tokenizer.normalizer.normalize_str("Héllò hôw are ü?"))

hello how are u?


In this example, since we picked the `bert-base-uncased` checkpoint, the normalization applied lowercasing and removed the accents. 


> ✏️ **Try it out!** Load a tokenizer from the `bert-base-cased` checkpoint and pass the same example to it. What are the main differences you can see between the cased and uncased versions of the tokenizer?


In [7]:
tokenizer2 = AutoTokenizer.from_pretrained("bert-base-cased")
print(
    tokenizer2.backend_tokenizer.normalizer.normalize_str("Héllò hôw are ü?")
)

Héllò hôw are ü?


So, there has been no change in the casing or accents by `bert-base-cased`! Let's dig deeper.

In [10]:
print(tokenizer2.backend_tokenizer.normalizer)

BertNormalizer(clean_text=True, handle_chinese_chars=True, strip_accents=None, lowercase=False)


Here👆 we can see that `strip_accents=None`, `lowercase=False` - that explains the behaviour. But, what does `clean_text` do here?

`clean_text=True` mainly does two things:

- **Removes certain control characters.**  Characters such as null (\x00) or other Unicode control characters are removed.<br> 
For example, conceptually:<br>
"hello\x00world" -> "helloworld"


- **Converts certain whitespace characters into a regular space (" ").** Whitespace characters are normalized to the ordinary ASCII space:<br>
"hello\tworld" -> "hello world"<br>"hello\nworld" -> "hello world"

Let's check:


In [19]:
print(tokenizer2.backend_tokenizer.normalizer.normalize_str("I\nam\nhere"))
print(tokenizer2.backend_tokenizer.normalizer.normalize_str("I\x00am\x00here"))

I am here
Iamhere


## Pre-tokenization

As we will see in the next sections, a tokenizer cannot be trained on raw text alone. Instead, we first need to split the texts into small entities, like words. That's where the pre-tokenization step comes in. As we saw in Chapter 2, a word-based tokenizer can simply split a raw text into words on whitespace and punctuation. Those words will be the boundaries of the subtokens the tokenizer can learn during its training.

To see how a fast tokenizer performs pre-tokenization, we can use the `pre_tokenize_str()` method of the `pre_tokenizer` attribute of the `tokenizer` object:

In [20]:
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(
    "Hello, how are  you?"
)

[('Hello', (0, 5)),
 (',', (5, 6)),
 ('how', (7, 10)),
 ('are', (11, 14)),
 ('you', (16, 19)),
 ('?', (19, 20))]

Notice how the tokenizer is already keeping track of the offsets, which is how it can give us the offset mapping we used in the previous section. Here the tokenizer ignores the two spaces and replaces them with just one, but the offset jumps between `are` and `you` to account for that.

Since we're using a BERT tokenizer, the pre-tokenization involves splitting on whitespace and punctuation. Other tokenizers can have different rules for this step. For example, if we use the GPT-2 tokenizer:

In [21]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(
    "Hello, how are  you?"
)

[('Hello', (0, 5)),
 (',', (5, 6)),
 ('Ġhow', (6, 10)),
 ('Ġare', (10, 14)),
 ('Ġ', (14, 15)),
 ('Ġyou', (15, 19)),
 ('?', (19, 20))]

...it has split on whitespace and punctuation as well, but it has kept the spaces and replace them with a `Ġ` symbol, enabling it to recover the original spaces if we decode the tokens.

Also note that unlike the BERT tokenizer, this tokenizer does not ignore the double space.

For a last example, let's have a look at the T5 tokenizer, which is based on the SentencePiece algorithm:


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("t5-small")
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(
    "Hello, how are  you?"
)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

[('▁Hello,', (0, 6)),
 ('▁how', (7, 10)),
 ('▁are', (11, 14)),
 ('▁you?', (16, 20))]

: 

Like the GPT-2 tokenizer, this one keeps spaces and replaces them with a specific token (`_`), but the T5 tokenizer only splits on whitespace, not punctuation. Also note that it added a space by default at the beginning of the sentence (before `Hello`) and ignored the double space between `are` and `you`.

Now that we've seen a little of how some different tokenizers process text, we can start to explore the underlying algorithms themselves. We'll begin with a quick look at the broadly widely applicable SentencePiece; then, over the next three sections, we'll examine how the three main algorithms used for subword tokenization work.